# Specific Functions
> These are functions only apply to the tables required

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

In [0]:
%run /Workspace/Repos/nikum.vedansh@gmail.com/formula-one-project/scripts/utils/helper_utils

In [0]:
def races_specific(df):
    df = df.drop("year")
    time_cols = ["time", "fp1_time", "fp2_time", "fp3_time", "quali_time", "sprint_time"]
    for col in time_cols:
        if col in df.columns:
            df = df.withColumn(col, clean_wall_clock(col))
    return df

In [0]:
def results_specific(df):
    df = df.withColumn("fastestlapspeed", F.col("fastestlapspeed").try_cast(DoubleType()))
    df = df.withColumn("fastestlap", F.col("fastestlap").try_cast(IntegerType()))
    df = df.withColumn("fastestlaptime", convert_time_to_ms("fastestlaptime"))
    df = df.drop("time")
    return df

In [0]:
def pit_stops_specific(df):
    df = df.withColumn("duration", convert_time_to_ms("duration"))
    df = df.withColumn("time", clean_wall_clock("time"))
    return df

In [0]:
def lap_times_specific(df):
    df = df.drop("time")
    return df

In [0]:
def qualifying_specific(df):
    for col in ["q1", "q2", "q3"]:
        if col in df.columns:
            df = df.withColumn(col, convert_time_to_ms(col))
    return df

In [0]:
def sprint_results_specific(df):
    df = df.withColumn("fastestlaptime", convert_time_to_ms("fastestlaptime"))
    df = df.withColumn("fastestlap", F.col("fastestlap").try_cast(IntegerType()))
    df = df.withColumn("rank", F.col("rank").try_cast(IntegerType()))
    df = df.drop("time")
    return df

In [0]:
def seasons_specific(df):
    df = df.withColumn("year", F.to_date(F.col("year").cast("string"), "yyyy"))
    return df

In [0]:
def safety_cars_specific(df):
    df = df.withColumn("cause",
        F.when(F.col("cause").rlike("(?i).*accident.*"), "Accident(s)")
         .when(F.col("cause").rlike("(?i).*stranded car.*"), "Stranded Car(s)")
         .when(F.col("cause").rlike("(?i).*debris.*"), "Debris")
         .otherwise(F.col("cause"))
    )
    return df

In [0]:
#Master to apply functions
def apply_specific_silver_transforms(df, table_name):
    f1_transform_map = {
        "race_data_races": races_specific,
        "race_data_results": results_specific,
        "race_data_pit_stops": pit_stops_specific,
        "race_data_lap_times": lap_times_specific,
        "race_data_qualifying": qualifying_specific,
        "race_data_sprint_results": sprint_results_specific,
        "race_data_seasons": seasons_specific,
        "race_events_safety_cars": safety_cars_specific
    }
    if table_name in f1_transform_map:
        df = f1_transform_map[table_name](df)
    return df